# Detección de ocupación de parqueaderos — flujo completo en Colab
Proyecto final de **Sistemas de Inteligencia Artificial**. Los 3 modelos (CNN, MiniDetector y Transformer) se diseñan capa por capa y se **entrenan desde cero** (sin pesos preentrenados).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.

Orden: instalar → descargar datos → preparar → entrenar CNN (SGD vs Adam) → Hill Climbing + estrategia evolutiva → reentrenar → asistente → detector → evaluar → pytest → app.

## 0. Obtener el código
Opción A: sube `parqueadero-ia.zip` al panel de archivos y descomprímelo. Opción B: clona tu repositorio.

In [ ]:
import os
# Opción A: zip subido a /content
if os.path.exists('/content/parqueadero-ia.zip') and not os.path.exists('/content/parqueadero-ia'):
    !unzip -q /content/parqueadero-ia.zip -d /content/
# Opción B: !git clone https://github.com/<tu_usuario>/parqueadero-ia.git /content/parqueadero-ia
%cd /content/parqueadero-ia
# rutas_locales.yaml es solo para el PC donde se desarrolló; en Colab se usan las rutas por defecto
!rm -f config/rutas_locales.yaml

In [ ]:
# (Opcional) guardar datos y resultados en Google Drive para no repetir la descarga
USAR_DRIVE = False
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['PARQUEADERO_DATOS'] = '/content/drive/MyDrive/parqueadero_datos'

## 1. Instalación
PyTorch ya viene en Colab; se instala el resto.

In [ ]:
!pip install -q opencv-python-headless scikit-learn streamlit matplotlib pandas pyyaml imageio imageio-ffmpeg tqdm pytest
import torch; print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO (cpu)')

## 2. Datos: CNRPark-EXT (~450 MB) y PKLot (~4.9 GB)
Si un enlace oficial falla, sube los archivos a mano y usa `--archivo-pklot` / `--archivo-cnr` (ver README).

In [ ]:
!python scripts/descargar_datos.py

In [ ]:
!python scripts/preparar_cnn.py
!python scripts/preparar_detector.py
!python herramientas/crear_video_pklot.py --parqueadero UFPR04

## 3. Modelo 1 — CNNEspacios: gradiente descendente (SGD con momentum vs Adam)

In [ ]:
!python entrenamiento/entrenar_cnn.py --experimento general

## 4. Optimización de hiperparámetros: Hill Climbing (básico y steepest ascent) y estrategia evolutiva (μ+λ)

In [ ]:
!python optimizacion/ejecutar_optimizacion.py

In [ ]:
from IPython.display import Image, display
display(Image('resultados/optimizacion_hiperparametros.png'))

### Reentrenar la CNN con la mejor configuración + experimentos de generalización

In [ ]:
!python entrenamiento/entrenar_cnn.py --usar-mejor-config
!python entrenamiento/entrenar_cnn.py --experimento parqueadero
!python entrenamiento/entrenar_cnn.py --experimento clima

## 5. Modelo 3 — TransformerAsistente (dataset sintético en español)

In [ ]:
!python asistente/generar_dataset.py
!python entrenamiento/entrenar_asistente.py

## 6. Modelo 2 — MiniDetector (cuadrícula 13×13, pérdida y NMS propios)

In [ ]:
!python entrenamiento/entrenar_detector.py

## 6b. Modelo 4 — DetectorUniversal (modo automático, sin JSON)
Se descargan de Roboflow Universe **solo datasets** (imágenes + anotaciones, CC BY 4.0), nunca modelos.
Necesitas una API key gratuita de Roboflow (Settings → API Keys).

In [ ]:
import os, getpass
if not os.environ.get('ROBOFLOW_API_KEY'):
    os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('API key de Roboflow: ')
!python scripts/descargar_roboflow.py
!python scripts/preparar_detector_universal.py
!python entrenamiento/entrenar_detector_universal.py
!python evaluacion/evaluar_detector_universal.py

## 7. Evaluación completa

In [ ]:
!python evaluacion/evaluar_cnn.py
!python evaluacion/evaluar_detector.py
!python evaluacion/evaluar_asistente.py
!python evaluacion/evaluar_video.py

In [ ]:
import glob
for p in sorted(glob.glob('resultados/*.png')):
    print(p); display(Image(p, width=900))

## 8. Pruebas unitarias

In [ ]:
!python -m pytest -q tests

## 9. Probar el pipeline y el asistente desde Python

In [ ]:
import cv2, json
from inferencia.pipeline import AnalizadorParqueadero
from inferencia.recorte import cargar_espacios
from asistente.asistente import Asistente
an = AnalizadorParqueadero('pesos/cnn_espacios.pth')
esp = cargar_espacios('config/espacios_UFPR04.json')
img = cv2.imread(sorted(glob.glob('ejemplos/*.jpg'))[0])
anotada, estado = an.analizar_imagen(img, esp)
cv2.imwrite('resultados/demo_anotada.jpg', anotada); display(Image('resultados/demo_anotada.jpg', width=900))
asis = Asistente('pesos/transformer_asistente.pth')
for q in ['¿hay cupo?', '¿qué tan lleno está?', '¿dónde puedo parquear?', '¿qué tan seguro estás?', 'dame un reporte']:
    print('>>', q, '
  ', asis.responder(q, estado)[0])

## 10. Ejecutar la app Streamlit desde Colab
Se expone con un túnel de Cloudflare (abre el enlace `trycloudflare.com` que aparece).

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!sleep 5 && ./cloudflared tunnel --url http://localhost:8501 2>&1 | grep -m1 -o 'https://.*trycloudflare.com'